In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)  # see all columns
pd.set_option('display.float_format', '{:.2f}'.format)

In [13]:
candidates = pd.read_csv('candidate_profiles.csv')
placements = pd.read_csv('placement_outcomes.csv')

print(candidates.shape)   # expect (500, 19)
print(placements.shape)   # expect (370, 24)

(500, 19)
(370, 24)


In [14]:
# === PHASE 3: EDA ON EACH TABLE ===

print("=== CANDIDATES ===")
print(candidates.info())
print(candidates.describe())
print(candidates.nunique())

print("\n=== PLACEMENTS ===")
print(placements.info())
print(placements.describe())
print(placements.nunique())

# Integrity checks
print("\n--- KEY INTEGRITY CHECKS ---")
print(f"Duplicate candidate_ids in candidates: {candidates['candidate_id'].duplicated().sum()}")
orphaned = ~placements['candidate_id'].isin(candidates['candidate_id'])
print(f"placement candidate_ids NOT found in candidates: {orphaned.sum()}")

=== CANDIDATES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   candidate_id              500 non-null    int64 
 1   first_name                500 non-null    object
 2   last_name                 500 non-null    object
 3   nationality               500 non-null    object
 4   job_function              500 non-null    object
 5   seniority_level           500 non-null    object
 6   years_of_experience       500 non-null    int64 
 7   expected_salary_gbp       500 non-null    int64 
 8   assessment_score          500 non-null    int64 
 9   key_skills                500 non-null    object
 10  english_proficiency       500 non-null    int64 
 11  remote_preference         500 non-null    object
 12  registration_date         500 non-null    object
 13  location                  500 non-null    object
 14  educati

In [15]:
# === PHASE 4: FIX DATA TYPES ===

candidates['registration_date'] = pd.to_datetime(candidates['registration_date'])
placements['start_date'] = pd.to_datetime(placements['start_date'])

placements['end_date'] = pd.to_datetime(placements['end_date'])

cat_cols = ['seniority_level', 'remote_preference', 'placement_status', 
            'offer_accepted', 'rehire_likelihood']

for col in cat_cols:
    if col in candidates.columns:
        candidates[col] = candidates[col].astype('category')
    if col in placements.columns:
        placements[col] = placements[col].astype('category')

print("Dtypes fixed. Confirming:")
print(candidates.dtypes)
print(placements.dtypes)

Dtypes fixed. Confirming:
candidate_id                         int64
first_name                          object
last_name                           object
nationality                         object
job_function                        object
seniority_level                   category
years_of_experience                  int64
expected_salary_gbp                  int64
assessment_score                     int64
key_skills                          object
english_proficiency                  int64
remote_preference                 category
registration_date           datetime64[ns]
location                            object
education_level                     object
availability_notice_days             int64
visa_status                         object
source_channel                      object
profile_status                      object
dtype: object
placement_id                          int64
candidate_id                          int64
placement_status                   category
time_to_hir

In [16]:
# === PHASE 5: LEFT JOIN MERGE ===

df = candidates.merge(placements, on='candidate_id', how='left')
print(f"Merged shape: {df.shape}")  # Expect (500, 42)

Merged shape: (500, 42)


In [17]:
# === PHASE 6: UNDERSTAND AND HANDLE NULLS ===

null_summary = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(pd.DataFrame({'null_count': null_summary, 'null_%': null_pct})[null_summary > 0])

# Fill placement nulls (unplaced candidates)
placement_cat_cols = ['placement_status', 'rehire_likelihood', 'offer_accepted']
for col in placement_cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('object').fillna('Not Placed')

placement_num_cols = ['offered_salary_gbp', 'time_to_hire_days', 
                      'client_satisfaction_score', 'interview_rounds']
for col in placement_num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df['notes'] = df['notes'].fillna('No notes') if 'notes' in df.columns else df.get('notes', 'No notes')
print("\nNulls after fill:")
print(df.isnull().sum()[df.isnull().sum() > 0])

                           null_count  null_%
placement_id                      130   26.00
placement_status                  130   26.00
time_to_hire_days                 130   26.00
offered_salary_gbp                130   26.00
offer_accepted                    130   26.00
client_satisfaction_score         130   26.00
rehire_likelihood                 130   26.00
placement_fee_gbp                 130   26.00
estimated_fee_value_gbp           130   26.00
interview_rounds                  130   26.00
hiring_manager                    130   26.00
recruiter_assigned                130   26.00
client_company                    130   26.00
client_industry                   130   26.00
role_title                        130   26.00
contract_type                     130   26.00
placement_region                  130   26.00
start_date                        130   26.00
end_date                          302   60.40
probation_outcome                 130   26.00
performance_rating                

In [18]:
# === PHASE 7: FEATURE ENGINEERING ===

df['salary_gap'] = df['offered_salary_gbp'] - df['expected_salary_gbp']
df['salary_match_ratio'] = df['offered_salary_gbp'] / df['expected_salary_gbp'].replace(0, np.nan)
df['skills_count'] = df['key_skills'].str.split(',').str.len()
df['days_since_registration'] = (pd.Timestamp.today() - df['registration_date']).dt.days
df['interview_efficiency'] = df['time_to_hire_days'] / df['interview_rounds'].replace(0, np.nan)
df['was_placed'] = df['placement_status'].apply(lambda x: 0 if x == 'Not Placed' else 1)

print("New features added:")
print(df[['salary_gap','salary_match_ratio','skills_count',
          'days_since_registration','interview_efficiency','was_placed']].describe())

New features added:
       salary_gap  salary_match_ratio  skills_count  days_since_registration  \
count      500.00              500.00        500.00                   500.00   
mean    -12375.00                0.75          1.00                   672.99   
std      24442.93                0.45          0.00                   308.60   
min    -115500.00                0.00          1.00                   119.00   
25%     -20750.00                0.00          1.00                   429.00   
50%       -750.00                0.98          1.00                   662.50   
75%       2000.00                1.05          1.00                   925.25   
max      11000.00                1.12          1.00                  1242.00   

       interview_efficiency  was_placed  
count                370.00      500.00  
mean                  13.94        0.74  
std                    9.14        0.44  
min                    1.25        0.00  
25%                    8.33        0.00  
50%    

In [19]:
# === PHASE 8: SAVE MASTER DATASET ===

df.to_csv('Recruitment_master.csv', index=False)
print(f"Saved Recruitment_master.csv — shape: {df.shape}")

Saved Recruitment_master.csv — shape: (500, 49)
